# Head Egocentric: 3 Spike Types (Any pass_99)

This notebook compares per-cell egocentric summaries from three completed runs:
- `head_all_spike`
- `head_simple_spike`
- `head_complex_spike`

Selection rule:
- keep a cell if it passes `pass_99` in **at least one** of: `head_all_spike`, `head_simple_spike`, `head_complex_spike`.

Then, generate per-cell figures that display all/SS/CS side by side.


In [ ]:

%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys
import importlib
import json

BASE_DIR = Path('/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4')
ROOT = BASE_DIR / 'data' / 'egocentric_tuning_carpenter_full'
MANIFEST_DIR = ROOT
DATA_ROOT = BASE_DIR / 'data'
SPATIAL_CACHE_ROOT = DATA_ROOT
FIGURES_ROOT = BASE_DIR / 'figures'
ANY_PASS_THRESHOLD = 99
ANY_PASS_SUFFIX = f'any{ANY_PASS_THRESHOLD}'
OUT_DIR = ROOT / f'head_{ANY_PASS_SUFFIX}_three_spike_compare'
EXPECTED_RUNS = ('head_all_spike', 'head_simple_spike', 'head_complex_spike')
RUNTIME_MERGED_DATA_FILENAME = 'manual_spike_detection_results.pkl'

if str(BASE_DIR.parent) not in sys.path:
    sys.path.insert(0, str(BASE_DIR.parent))

import miniVI_PlaceCell_analysis_V4.notebooks_HPC.run_egocentric_plot_cells_head_three_spike_any99 as head_runner
import miniVI_PlaceCell_analysis_V4.notebooks_HPC.pf_split_group_plotting as pfsp
import miniVI_PlaceCell_analysis_V4.notebooks_HPC.egocentric_refined_config as refined_cfg

REFINED_ANIMALS = tuple(refined_cfg.ANIMALS)


def check_cluster_results_ready(*, raise_on_missing=False):
    missing = []
    required = [MANIFEST_DIR / 'manifest.json']
    for run_name in EXPECTED_RUNS:
        per_cell_dir = ROOT / run_name / 'per_cell_results'
        required.append(per_cell_dir)
    for path in required:
        if not path.exists():
            missing.append(path)
    for run_name in EXPECTED_RUNS:
        per_cell_dir = ROOT / run_name / 'per_cell_results'
        if per_cell_dir.exists() and not any(per_cell_dir.glob('cell_*.npz')):
            missing.append(per_cell_dir / 'cell_*.npz')
    if missing:
        msg = 'Missing copied cluster result inputs:\n' + '\n'.join(f'  {m}' for m in missing)
        if raise_on_missing:
            raise FileNotFoundError(msg)
        print('[WARN]', msg)
        return False
    print('[OK] Copied cluster result inputs found.')
    return True

print('Runner:', head_runner.__name__ + '.main')
print('Plot module:', pfsp.__name__)
print('Refined animals:', len(REFINED_ANIMALS), REFINED_ANIMALS)
print('Runtime data filename:', RUNTIME_MERGED_DATA_FILENAME)
print('Cluster result root:', ROOT)
print('Spatial cache root:', SPATIAL_CACHE_ROOT)
print('Threshold:', ANY_PASS_THRESHOLD)
print('Output:', OUT_DIR)
_ = check_cluster_results_ready(raise_on_missing=False)


In [2]:
first_n_minutes = 20.0
split_map_bin_size_cm = 5.0
split_preferred_angle_source = 'empirical'  # use 'empirical' or 'fit' 
stats_pref_reference = 'all'  # shared control for column 5/6 SS/CS split maps + PF stats ('all' or 'matching_metric')
stats_joint_valid_bins = False  # use only bins valid in both pref/non-pref maps for stats
split_preferred_half_width_deg = 50.0  # essential: preferred-angle half-width (deg)
col3_emp_arrow_length_scale = 2.0  # column 3 empirical-arrow length scale
binpolar_render_style = 'line'  # columns 7/8 mini-polar style: 'fan' or 'line'
pf_overlay_area_threshold = 0.5  # PF overlay remap threshold (coarse bin overlap ratio)
hd_vel_corr_method = 'mean_raw_dot'  # 'normalized_dot' or 'mean_raw_dot'

# Shared occupancy threshold (seconds), used for both egocentric split-map and placecell-map occupancy gating
occupancy_threshold_s = 0.2
occupancy_threshold_split_s = 0.2  # column 5/6 pref/nonpref split occupancy threshold (s)

# Valid spatial-bin / tuning gates (exposed for easy tuning)
valid_bin_min_occupied_angle_bins = 5  # bin valid if occupied-angle-count > this value
valid_bin_min_mean_rate_hz = 0.5  # bin valid if mean rate exceeds this threshold (Hz)
valid_bin_min_spikes = 2  # bin valid only if spike count in the bin is at least this value
tuning_min_valid_bins = 5  # min valid bins required for tuning/pass decisions

# Comparison-figure sizing: subplot width = groups_in_column * group_width_in (inches)
group_width_in_combined = 0.26
group_width_in_ds = group_width_in_combined * (34.0 / 22.0)  # match overall width to combined figure
group_width_in = group_width_in_combined  # optional shared default


## Per cell summary

In [ ]:

import importlib
importlib.reload(head_runner)
importlib.reload(pfsp)
importlib.reload(refined_cfg)

check_cluster_results_ready(raise_on_missing=True)

params = {
    'base_dir': ROOT,
    'manifest_dir': MANIFEST_DIR,
    'output_dir': OUT_DIR,
    'all_run': 'head_all_spike',
    'ss_run': 'head_simple_spike',
    'cs_run': 'head_complex_spike',
    'data_root': DATA_ROOT,
    'merged_data_filename': RUNTIME_MERGED_DATA_FILENAME,
    'spatial_cache_root': SPATIAL_CACHE_ROOT,
    'figures_root': FIGURES_ROOT,
    'direction_mode': 'head',
    'first_n_minutes': first_n_minutes,
    'split_map_bin_size_cm': split_map_bin_size_cm,
    'split_preferred_angle_source': split_preferred_angle_source,
    'split_preferred_half_width_deg': split_preferred_half_width_deg,
    'col3_emp_arrow_length_scale': col3_emp_arrow_length_scale,
    'binpolar_render_style': binpolar_render_style,
    'pf_overlay_area_threshold': pf_overlay_area_threshold,
    'stats_pref_reference': stats_pref_reference,
    'stats_joint_valid_bins': stats_joint_valid_bins,
    'any_pass_threshold': ANY_PASS_THRESHOLD,
    'hd_vel_corr_method': hd_vel_corr_method,
    'occupancy_threshold_s': occupancy_threshold_s,
    'occupancy_threshold_split_s': occupancy_threshold_split_s,
    'valid_bin_min_occupied_angle_bins': valid_bin_min_occupied_angle_bins,
    'valid_bin_min_mean_rate_hz': valid_bin_min_mean_rate_hz,
    'valid_bin_min_spikes': valid_bin_min_spikes,
    'tuning_min_valid_bins': tuning_min_valid_bins,
    'save_formats': ('svg',),
}

run_args = [
    '--base-dir', str(params['base_dir']),
    '--manifest-dir', str(params['manifest_dir']),
    '--output-dir', str(params['output_dir']),
    '--all-run', params['all_run'],
    '--ss-run', params['ss_run'],
    '--cs-run', params['cs_run'],
    '--data-root', str(params['data_root']),
    '--merged-data-filename', str(params['merged_data_filename']),
    '--spatial-cache-root', str(params['spatial_cache_root']),
    '--figures-root', str(params['figures_root']),
    '--direction-mode', params['direction_mode'],
    '--first-n-minutes', str(params['first_n_minutes']),
    '--split-map-bin-size-cm', str(params['split_map_bin_size_cm']),
    '--split-preferred-angle-source', str(params['split_preferred_angle_source']),
    '--split-preferred-half-width-deg', str(params['split_preferred_half_width_deg']),
    '--col3-emp-arrow-length-scale', str(params['col3_emp_arrow_length_scale']),
    '--binpolar-render-style', str(params['binpolar_render_style']),
    '--pf-overlay-area-threshold', str(params['pf_overlay_area_threshold']),
    '--stats-pref-reference', str(params['stats_pref_reference']),
    '--stats-joint-valid-bins', str(params['stats_joint_valid_bins']).lower(),
    '--any-pass-threshold', str(params['any_pass_threshold']),
    '--hd-vel-corr-method', str(params['hd_vel_corr_method']),
    '--occupancy-threshold-s', str(params['occupancy_threshold_s']),
    '--occupancy-threshold-split-s', str(params['occupancy_threshold_split_s']),
    '--valid-bin-min-occupied-angle-bins', str(params['valid_bin_min_occupied_angle_bins']),
    '--valid-bin-min-mean-rate-hz', str(params['valid_bin_min_mean_rate_hz']),
    '--valid-bin-min-spikes', str(params['valid_bin_min_spikes']),
    '--tuning-min-valid-bins', str(params['tuning_min_valid_bins']),
    '--save-formats', *params['save_formats'],
]

print('Calling run_head_any_threshold with:')
print('  ' + ' '.join(run_args))
head_runner.main(run_args)
print('Figures saved to:', OUT_DIR)


In [ ]:
import pandas as pd
from pandas.errors import EmptyDataError

per_cell_root = OUT_DIR / 'per_cell_summary'
manifest_csv = per_cell_root / f'egocentric_per_cell_plot_manifest_{ANY_PASS_SUFFIX}_3spike.csv'
skip_csv = per_cell_root / f'egocentric_per_cell_plot_skipped_{ANY_PASS_SUFFIX}_3spike.csv'

if not manifest_csv.exists():
    raise FileNotFoundError(f'Missing manifest: {manifest_csv}')

try:
    df = pd.read_csv(manifest_csv)
except EmptyDataError:
    df = pd.DataFrame()

print('Manifest rows:', len(df))
if df.empty:
    print('Manifest CSV is empty (no selected cells for this threshold).')
else:
    print('By category:')
    print(df['category'].value_counts(dropna=False))

    pass_cols = [
        f'pass{ANY_PASS_THRESHOLD}_all',
        f'pass{ANY_PASS_THRESHOLD}_ss',
        f'pass{ANY_PASS_THRESHOLD}_cs',
    ]
    missing_cols = [c for c in pass_cols if c not in df.columns]
    if missing_cols:
        raise KeyError(f'Missing threshold columns in manifest: {missing_cols}')

    print(f'Any-pass contributors (pass_{ANY_PASS_THRESHOLD}):')
    print(df[pass_cols].sum())

n_svg = len(list(per_cell_root.rglob('*.svg')))
n_png = len(list(per_cell_root.rglob('*.png')))
print('SVG files:', n_svg)
print('PNG files:', n_png)
if n_png > 0:
    raise AssertionError('Expected SVG-only outputs, but PNG files were found.')
print('Manifest CSV:', manifest_csv)
print('Skip CSV:', skip_csv)


## Behavior Head-Direction Tuning (Refined 8 Animals)

Overall behavior head-direction occupancy tuning using the same local refined animal list and runtime data source as this analysis run:
- `direction_mode='head'`
- `n_angle_bins=10`
- `speed in [3, 60] cm/s`
- `first_n_minutes=first_n_minutes`


In [ ]:

from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
import miniVI_PlaceCell_analysis_V4.utils.placecell_pipeline as _pcp_behavior

ANIMALS = list(REFINED_ANIMALS)

behavior_config = refined_cfg.build_refined_config(
    project_root=BASE_DIR,
    data_root=DATA_ROOT,
    figures_root=FIGURES_ROOT,
    force_recompute=False,
)
behavior_config.merged_data_filename = RUNTIME_MERGED_DATA_FILENAME

N_ANGLE_BINS = 10
FIRST_N_MINUTES = first_n_minutes
SPEED_MIN = 3.0
SPEED_MAX = 60.0
ARENA_SIZE_CM = (35.5, 20.0)

ANGLE_EDGES = np.linspace(-np.pi, np.pi, N_ANGLE_BINS + 1, dtype=float)
ANGLE_CENTERS = 0.5 * (ANGLE_EDGES[:-1] + ANGLE_EDGES[1:])
THETA_DISP = np.mod(ANGLE_CENTERS - (np.pi / 2.0), 2.0 * np.pi)
BAR_WIDTH = (2.0 * np.pi) / float(N_ANGLE_BINS)


def load_merged_data(animal_dir: Path) -> dict:
    return _pcp_behavior._load_merged_data(animal_dir, behavior_config)


def wrap_to_pi(angle_rad):
    return (np.asarray(angle_rad, dtype=float) + np.pi) % (2.0 * np.pi) - np.pi


def coerce_hd_to_radians_wrapped(hd_values):
    hd_arr = np.asarray(hd_values, dtype=float).reshape(-1)
    finite = hd_arr[np.isfinite(hd_arr)]
    if finite.size == 0:
        return np.full(hd_arr.shape, np.nan, dtype=float)
    if float(np.nanmax(np.abs(finite))) > (np.pi * 1.25):
        return wrap_to_pi(np.deg2rad(hd_arr))
    return wrap_to_pi(hd_arr)


def build_head_direction_valid_mask(
    *,
    x_frames,
    y_frames,
    hd_frames,
    speed_frames,
    frame_rate,
    first_n_minutes,
    arena_size_cm,
    speed_min_cm_s,
    speed_max_cm_s,
):
    x_arr = np.asarray(x_frames, dtype=float).reshape(-1)
    y_arr = np.asarray(y_frames, dtype=float).reshape(-1)
    speed_arr = np.asarray(speed_frames, dtype=float).reshape(-1)
    dir_frames = coerce_hd_to_radians_wrapped(hd_frames)

    width_cm, height_cm = float(arena_size_cm[0]), float(arena_size_cm[1])
    in_bounds = (
        np.isfinite(x_arr) & np.isfinite(y_arr)
        & (x_arr >= 0.0) & (x_arr <= width_cm)
        & (y_arr >= 0.0) & (y_arr <= height_cm)
    )
    valid = (
        np.isfinite(x_arr)
        & np.isfinite(y_arr)
        & np.isfinite(dir_frames)
        & np.isfinite(speed_arr)
        & in_bounds
        & (speed_arr >= float(speed_min_cm_s))
        & (speed_arr <= float(speed_max_cm_s))
    )

    if first_n_minutes is not None:
        cutoff = int(np.floor(float(first_n_minutes) * 60.0 * float(frame_rate)))
        cutoff = max(1, min(int(x_arr.size), cutoff))
        if cutoff < int(x_arr.size):
            valid[cutoff:] = False

    return np.asarray(dir_frames, dtype=float), np.asarray(valid, dtype=bool)


n_cols = 4
n_rows = int(math.ceil(len(ANIMALS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, subplot_kw={'projection': 'polar'}, figsize=(16.0, 4.0 * n_rows), dpi=150)
axes = np.asarray(axes, dtype=object).reshape(-1)

for ax, animal_id in zip(axes, ANIMALS):
    merged = load_merged_data(DATA_ROOT / animal_id)

    x_frames = np.asarray(merged['x_neural'], dtype=float)
    y_frames = np.asarray(merged['y_neural'], dtype=float)
    hd_frames = np.asarray(merged.get('hd_angles_neural', np.full_like(x_frames, np.nan)), dtype=float)
    speed_frames = np.asarray(merged['speed'], dtype=float)
    frame_rate = float(merged['frame_rate'])

    dir_frames, valid_mask = build_head_direction_valid_mask(
        x_frames=x_frames,
        y_frames=y_frames,
        hd_frames=hd_frames,
        speed_frames=speed_frames,
        frame_rate=frame_rate,
        first_n_minutes=FIRST_N_MINUTES,
        arena_size_cm=ARENA_SIZE_CM,
        speed_min_cm_s=SPEED_MIN,
        speed_max_cm_s=SPEED_MAX,
    )

    dirs = np.asarray(dir_frames[valid_mask], dtype=float)
    dirs = dirs[np.isfinite(dirs)]

    counts, _ = np.histogram(dirs, bins=ANGLE_EDGES)
    occ_s = np.asarray(counts, dtype=float) / float(frame_rate)
    total_s = float(np.sum(occ_s))
    occ_frac = (occ_s / total_s) if total_s > 0 else np.zeros_like(occ_s)

    ax.bar(
        THETA_DISP,
        occ_frac,
        width=BAR_WIDTH,
        align='center',
        color='#4C78A8',
        edgecolor='white',
        linewidth=0.4,
        alpha=0.85,
        zorder=2,
    )

    if dirs.size > 0:
        vec = np.mean(np.exp(1j * dirs))
        if np.isfinite(np.real(vec)) and np.isfinite(np.imag(vec)):
            mrl = float(np.clip(np.abs(vec), 0.0, 1.0))
            theta_mean = float(np.mod(np.angle(vec) - (np.pi / 2.0), 2.0 * np.pi))
            r_top = float(max(np.max(occ_frac) * 1.05, 1e-6))
            ax.annotate(
                '',
                xy=(theta_mean, r_top * mrl),
                xytext=(theta_mean, 0.0),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.0),
                zorder=4,
            )
            ax.text(0.03, 0.95, f'MRL={mrl:.3f}', transform=ax.transAxes, ha='left', va='top', fontsize=8)

    ax.set_theta_zero_location('N')
    ax.set_theta_direction(1)
    ax.set_xticks(np.deg2rad([0, 90, 180, 270]))
    ax.set_xticklabels(['N', 'W', 'S', 'E'], fontsize=8)
    ax.set_yticklabels([])
    ax.set_ylim(0.0, float(max(np.max(occ_frac) * 1.25, 0.05)))
    ax.grid(alpha=0.25, linewidth=0.6)
    ax.set_title(animal_id, fontsize=10, pad=12)
    ax.text(0.03, 0.06, f'valid={int(np.sum(valid_mask))} frames', transform=ax.transAxes, ha='left', va='bottom', fontsize=7)

for ax in axes[len(ANIMALS):]:
    ax.set_axis_off()

minutes_label = 'full session' if FIRST_N_MINUTES is None else f'first {float(FIRST_N_MINUTES):g} min'
fig.suptitle(
    f'Behavior Head-Direction Occupancy Tuning ({minutes_label}; speed 3-60 cm/s; 10 angle bins)',
    fontsize=12,
    y=0.98,
)
plt.tight_layout(rect=[0.0, 0.0, 1.0, 0.95])
plt.show()


## Percentage of any-pass cells contributed by each category

In [6]:
import importlib
importlib.reload(pfsp)

pf_split_csv = OUT_DIR / 'per_cell_summary' / f'egocentric_pf_split_stats_{ANY_PASS_SUFFIX}_3spike.csv'
summary_dir = OUT_DIR / 'summary_stats'
summary_dir.mkdir(parents=True, exist_ok=True)

any_pass_result = pfsp.plot_any_pass_category_contribution(
    pf_split_csv=pf_split_csv,
    out_dir=summary_dir,
    any_pass_threshold=ANY_PASS_THRESHOLD,
    any_pass_suffix=ANY_PASS_SUFFIX,
    style_opts={
        'font_family': 'Arial',
        'font_size': 6.0,
        'axes_labelsize': 6.0,
        'axes_titlesize': 6.0,
        'tick_labelsize': 5.0,
        'legend_fontsize': 5.0,
        'axes_linewidth': 0.5,
        'remove_top_right_spines': True,
    },
    show=True,
)

print('Saved any-pass figure path:', Path(any_pass_result['figure']).resolve())
print('Saved any-pass stats CSV:', Path(any_pass_result['stats_csv']).resolve())


Saved any-pass figure path: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/summary_stats/any_pass_category_contribution_any100.svg
Saved any-pass stats CSV: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/summary_stats/any_pass_category_contribution_any100.csv


/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/notebooks_HPC/pf_split_group_plotting.py:2052: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## SS vs CS Valid-Bin Mean MRL Comparison

Comparison uses per-cell values shown in column 8 titles (Valid-bin mean MRL), with inclusion criterion:
- `valid_bin_mrl_n_ss >= 4` and `valid_bin_mrl_n_cs >= 4`


In [7]:
# import numpy as np
# import pandas as pd
# from pandas.errors import EmptyDataError
# import matplotlib.pyplot as plt
# from scipy import stats
# from matplotlib.lines import Line2D

# manifest_csv = OUT_DIR / 'per_cell_summary' / f'egocentric_per_cell_plot_manifest_{ANY_PASS_SUFFIX}_3spike.csv'
# try:
#     df = pd.read_csv(manifest_csv)
# except EmptyDataError:
#     raise RuntimeError(
#         f'Manifest CSV is empty: {manifest_csv}. '
#         'No cells were selected at the current threshold.'
#     )

# required_cols = [
#     'category',
#     'valid_bin_mean_mrl_ss',
#     'valid_bin_mean_mrl_cs',
#     'valid_bin_mrl_n_ss',
#     'valid_bin_mrl_n_cs',
# ]
# missing = [c for c in required_cols if c not in df.columns]
# if missing:
#     raise KeyError(
#         f"Missing required manifest columns: {missing}. "
#         "Re-run the plotting cell to regenerate manifest with these fields."
#     )

# for c in ['valid_bin_mean_mrl_ss', 'valid_bin_mean_mrl_cs', 'valid_bin_mrl_n_ss', 'valid_bin_mrl_n_cs']:
#     df[c] = pd.to_numeric(df[c], errors='coerce')

# cmp_df = df[
#     np.isfinite(df['valid_bin_mean_mrl_ss'])
#     & np.isfinite(df['valid_bin_mean_mrl_cs'])
#     & (df['valid_bin_mrl_n_ss'] >= 4)
#     & (df['valid_bin_mrl_n_cs'] >= 4)
# ].copy()

# if cmp_df.empty:
#     raise RuntimeError('No cells pass SS/CS valid-bin filter (>=4 each).')

# preferred_order = ['CSplus', 'CSminus', 'all-nonPLC']
# categories = [c for c in preferred_order if c in set(cmp_df['category'])]
# for c in sorted(set(cmp_df['category'])):
#     if c not in categories:
#         categories.append(c)


# def paired_pvalue(x, y):
#     x = np.asarray(x, dtype=float)
#     y = np.asarray(y, dtype=float)
#     mask = np.isfinite(x) & np.isfinite(y)
#     x = x[mask]
#     y = y[mask]
#     n = x.size
#     if n < 2:
#         return np.nan, 'n<2'
#     if np.allclose(x - y, 0.0, atol=1e-12, rtol=0.0):
#         return 1.0, 'all_equal'
#     try:
#         p = float(stats.wilcoxon(x, y, alternative='two-sided', zero_method='wilcox').pvalue)
#         return p, 'wilcoxon'
#     except Exception:
#         p = float(stats.ttest_rel(x, y, nan_policy='omit').pvalue)
#         return p, 'ttest_rel'


# fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2), dpi=170, constrained_layout=True)
# ax_cat, ax_pool = axes

# # -------- Subplot 1: per-category --------
# cat_ns = []
# for i, cat in enumerate(categories):
#     sub = cmp_df.loc[cmp_df['category'] == cat].copy()
#     ss = sub['valid_bin_mean_mrl_ss'].to_numpy(dtype=float)
#     cs = sub['valid_bin_mean_mrl_cs'].to_numpy(dtype=float)
#     n = int(min(ss.size, cs.size))
#     cat_ns.append(n)
#     if n <= 0:
#         continue

#     for j in range(n):
#         ax_cat.plot([i - 0.12, i + 0.12], [ss[j], cs[j]], color='0.75', lw=0.8, alpha=0.8, zorder=1)

#     ax_cat.scatter(np.full(n, i - 0.12), ss, s=22, c='#1F77B4', edgecolors='none', alpha=0.95, zorder=2)
#     ax_cat.scatter(np.full(n, i + 0.12), cs, s=22, c='#D62728', edgecolors='none', alpha=0.95, zorder=2)

#     mean_ss = float(np.nanmean(ss))
#     mean_cs = float(np.nanmean(cs))
#     ax_cat.plot([i - 0.19, i - 0.05], [mean_ss, mean_ss], color='#1F77B4', lw=2.4, zorder=3)
#     ax_cat.plot([i + 0.05, i + 0.19], [mean_cs, mean_cs], color='#D62728', lw=2.4, zorder=3)

#     p_val, test_name = paired_pvalue(ss, cs)
#     if np.isfinite(p_val):
#         y_txt = max(np.nanmax(ss), np.nanmax(cs)) + 0.05
#         ax_cat.text(i, y_txt, f'p={p_val:.3g} ({test_name})', ha='center', va='bottom', fontsize=7)

# xticklabels = [f'{cat} (n={n})' for cat, n in zip(categories, cat_ns)]
# ax_cat.set_xticks(np.arange(len(categories), dtype=float))
# ax_cat.set_xticklabels(xticklabels, fontsize=8)
# ax_cat.set_ylabel('Valid-bin mean MRL', fontsize=9)
# ax_cat.set_title('SS vs CS by category (filter: SS>=4 bins and CS>=4 bins)', fontsize=10)
# ax_cat.grid(axis='y', alpha=0.25, linewidth=0.6)

# legend_handles = [
#     Line2D([0], [0], marker='o', linestyle='none', color='none', markerfacecolor='#1F77B4', markeredgecolor='none', markersize=6, label='SS'),
#     Line2D([0], [0], marker='o', linestyle='none', color='none', markerfacecolor='#D62728', markeredgecolor='none', markersize=6, label='CS'),
#     Line2D([0], [0], color='0.75', lw=1.0, label='paired cell'),
# ]
# ax_cat.legend(handles=legend_handles, loc='best', fontsize=8, frameon=False)

# # -------- Subplot 2: pooled --------
# ss_all = cmp_df['valid_bin_mean_mrl_ss'].to_numpy(dtype=float)
# cs_all = cmp_df['valid_bin_mean_mrl_cs'].to_numpy(dtype=float)
# mask_all = np.isfinite(ss_all) & np.isfinite(cs_all)
# ss_all = ss_all[mask_all]
# cs_all = cs_all[mask_all]
# n_all = int(ss_all.size)

# for j in range(n_all):
#     ax_pool.plot([0.0, 1.0], [ss_all[j], cs_all[j]], color='0.75', lw=0.8, alpha=0.8, zorder=1)

# ax_pool.scatter(np.zeros(n_all), ss_all, s=24, c='#1F77B4', edgecolors='none', alpha=0.95, zorder=2)
# ax_pool.scatter(np.ones(n_all), cs_all, s=24, c='#D62728', edgecolors='none', alpha=0.95, zorder=2)

# mean_ss_all = float(np.nanmean(ss_all)) if n_all > 0 else np.nan
# mean_cs_all = float(np.nanmean(cs_all)) if n_all > 0 else np.nan
# if np.isfinite(mean_ss_all):
#     ax_pool.plot([-0.08, 0.08], [mean_ss_all, mean_ss_all], color='#1F77B4', lw=2.6, zorder=3)
# if np.isfinite(mean_cs_all):
#     ax_pool.plot([0.92, 1.08], [mean_cs_all, mean_cs_all], color='#D62728', lw=2.6, zorder=3)

# p_pool, pool_test = paired_pvalue(ss_all, cs_all)
# title_pool = f'Pooled SS vs CS (n={n_all})'
# if np.isfinite(p_pool):
#     title_pool += f' | p={p_pool:.3g} ({pool_test})'
# ax_pool.set_title(title_pool, fontsize=10)
# ax_pool.set_xticks([0.0, 1.0])
# ax_pool.set_xticklabels(['SS', 'CS'], fontsize=9)
# ax_pool.set_ylabel('Valid-bin mean MRL', fontsize=9)
# ax_pool.grid(axis='y', alpha=0.25, linewidth=0.6)

# all_plot_vals = np.concatenate([ss_all, cs_all]) if n_all > 0 else np.array([0.0])
# ymax = float(np.nanmax(all_plot_vals)) if np.any(np.isfinite(all_plot_vals)) else 1.0
# ymax = max(0.25, min(1.05, ymax + 0.12))
# for ax in (ax_cat, ax_pool):
#     ax.set_ylim(0.0, ymax)

# summary_dir = OUT_DIR / 'summary_stats'
# summary_dir.mkdir(parents=True, exist_ok=True)
# out_svg = summary_dir / 'egocentric_valid_bin_mean_mrl_ss_vs_cs.svg'
# fig.savefig(out_svg, format='svg', bbox_inches='tight')
# plt.show()

# print('Included cells:', len(cmp_df))
# print('Included by category:')
# print(cmp_df['category'].value_counts())
# print('Saved:', out_svg)


## PF-Split CS+ statistics figures

In [ ]:
import importlib
importlib.reload(pfsp)

# PF-Split group figures (multi-group)
pf_split_csv = OUT_DIR / 'per_cell_summary' / f'egocentric_pf_split_stats_{ANY_PASS_SUFFIX}_3spike.csv'
summary_dir = OUT_DIR / 'summary_stats'
summary_dir.mkdir(parents=True, exist_ok=True)

# Group definitions
# EB tuned = cells actually saved under per_cell_summary/pass_{ANY_PASS_SUFFIX}
pf_group_specs = [
    {
        'id': f'csplus_eb_tuned_pass{ANY_PASS_THRESHOLD}',
        'title': f'CS+ PF split statistics (EB tuned pass{ANY_PASS_THRESHOLD})',
        'category_in': ['CSplus'],
        'is_place_cell': True,
        'selected_in_pass_any_folder': True,
        'regions': ['primary', 'secondary', 'combined', 'outside_combined', 'all_bins'],
    },
    {
        'id': 'csplus_all',
        'title': 'CS+ PF split statistics (all place cells)',
        'category_in': ['CSplus'],
        'is_place_cell': True,
        'regions': ['primary', 'secondary', 'combined', 'outside_combined', 'all_bins'],
    },
    {
        'id': 'csminus_all',
        'title': 'CS- PF split statistics (all place cells)',
        'category_in': ['CSminus'],
        'is_place_cell': True,
        'mrl_panel_mode': 'ss_only',
        'regions': ['primary', 'secondary', 'combined', 'outside_combined', 'all_bins'],
    },
    {
        'id': f'all_eb_tuned_pass{ANY_PASS_THRESHOLD}_all_bins_only',
        'title': 'All EB tuned cells PF split statistics (CS+ PLC + nonPLC)',
        'category_in': ['CSplus', 'all-nonPLC'],
        'selected_in_pass_any_folder': True,
        'regions': ['all_bins'],
    },
    {
        'id': f'nonplc_eb_tuned_pass{ANY_PASS_THRESHOLD}',
        'title': 'Non-place cells PF split statistics (EB tuned)',
        'category_in': ['all-nonPLC'],
        'is_place_cell': False,
        'selected_in_pass_any_folder': True,
        'regions': ['all_bins'],
    },
]

style_opts = {
    'font_family': 'Arial',
    'font_size': 6.0,
    'axes_labelsize': 6.0,
    'axes_titlesize': 6.0,
    'tick_labelsize': 5.0,
    'legend_fontsize': 5.0,
    'axes_linewidth': 0.5,
    'remove_top_right_spines': True,
    'show_p_after_sig': False,
    'group_width_in': group_width_in_combined,
}

stats_opts = {
    'correction_method': 'bh_fdr',
    'multiple_comparison_panel': 'ss_cs_rate_merged',
}

result = pfsp.plot_pf_split_group_figures(
    pf_split_csv=pf_split_csv,
    out_dir=summary_dir,
    any_pass_threshold=ANY_PASS_THRESHOLD,
    any_pass_suffix=ANY_PASS_SUFFIX,
    group_specs=pf_group_specs,
    style_opts=style_opts,
    stats_opts=stats_opts,
    show=True,
)

print('Saved PF split figures:')
for p in result.get('figures', []):
    print(' ', p)
print('Saved PF split stats CSVs:')
for p in result.get('stats_csvs', []):
    print(' ', p)


In [9]:
import importlib
importlib.reload(pfsp)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pf_split_csv = OUT_DIR / 'per_cell_summary' / f'egocentric_pf_split_stats_{ANY_PASS_SUFFIX}_3spike.csv'
summary_dir = OUT_DIR / 'summary_stats'
summary_dir.mkdir(parents=True, exist_ok=True)

inside_color = '#C2185B'
outside_color = '#7A7A7A'
id_cols = ['animal_id', 'cell_idx', 'cell_num']

pf_df = pd.read_csv(pf_split_csv)
for col in ['cell_idx', 'cell_num', 'preferred_mean', 'nonpreferred_mean']:
    pf_df[col] = pd.to_numeric(pf_df[col], errors='coerce')
pf_df['is_place_cell'] = pfsp._coerce_bool_series(pf_df['is_place_cell'])
pf_df['selected_in_pass_any_folder'] = pfsp._coerce_bool_series(pf_df['selected_in_pass_any_folder'])

plot_df = pf_df.loc[
    (pf_df['category'].astype(str) == 'CSplus')
    & (pf_df['is_place_cell'] == True)
    & (pf_df['selected_in_pass_any_folder'] == True)
    & (pf_df['region'].astype(str).isin(['combined', 'outside_combined']))
].copy()
if plot_df.empty:
    raise RuntimeError('No CS+ EB-tuned PF-split rows found for combined/outside_combined.')

style_opts = {
    'font_family': 'Arial',
    'font_size': 6.0,
    'axes_labelsize': 6.0,
    'axes_titlesize': 6.0,
    'tick_labelsize': 5.0,
    'legend_fontsize': 5.0,
    'axes_linewidth': 0.5,
    'remove_top_right_spines': True,
}
pfsp._apply_style(style_opts)


def _metric_region_frame(metric_name):
    sub = plot_df.loc[
        plot_df['metric'].astype(str) == str(metric_name),
        id_cols + ['region', 'preferred_mean'],
    ].copy()
    sub = sub.drop_duplicates(subset=id_cols + ['region'])
    if sub.empty:
        return pd.DataFrame(columns=id_cols + ['in_val', 'out_val'])
    wide = (
        sub.pivot_table(index=id_cols, columns='region', values='preferred_mean', aggfunc='first')
        .reset_index()
        .rename(columns={'combined': 'in_val', 'outside_combined': 'out_val'})
    )
    for col in ['in_val', 'out_val']:
        if col not in wide.columns:
            wide[col] = np.nan
    wide['in_val'] = pd.to_numeric(wide['in_val'], errors='coerce')
    wide['out_val'] = pd.to_numeric(wide['out_val'], errors='coerce')
    mask = np.isfinite(wide['in_val'].to_numpy(dtype=float)) & np.isfinite(wide['out_val'].to_numpy(dtype=float))
    return wide.loc[mask, id_cols + ['in_val', 'out_val']].copy()


def _ds_region_frame(metric_name, *, slow_use_abs=True, abs_ds=False):
    frames = []
    for region_name, region_df in plot_df.groupby(plot_df['region'].astype(str)):
        metric_df = pfsp._metric_ds_frame(region_df.copy(), metric_name, id_cols, slow_use_abs=slow_use_abs)
        if metric_df.empty:
            continue
        metric_df = metric_df.copy()
        if bool(abs_ds):
            metric_df['ds'] = np.abs(pd.to_numeric(metric_df['ds'], errors='coerce').to_numpy(dtype=float))
        metric_df['region'] = str(region_name)
        frames.append(metric_df)
    if not frames:
        return pd.DataFrame(columns=id_cols + ['in_val', 'out_val'])
    merged = pd.concat(frames, ignore_index=True)
    wide = (
        merged.pivot_table(index=id_cols, columns='region', values='ds', aggfunc='first')
        .reset_index()
        .rename(columns={'combined': 'in_val', 'outside_combined': 'out_val'})
    )
    for col in ['in_val', 'out_val']:
        if col not in wide.columns:
            wide[col] = np.nan
    wide['in_val'] = pd.to_numeric(wide['in_val'], errors='coerce')
    wide['out_val'] = pd.to_numeric(wide['out_val'], errors='coerce')
    mask = np.isfinite(wide['in_val'].to_numpy(dtype=float)) & np.isfinite(wide['out_val'].to_numpy(dtype=float))
    return wide.loc[mask, id_cols + ['in_val', 'out_val']].copy()


def _paired_values(pair_df):
    if pair_df.empty:
        return np.array([], dtype=float), np.array([], dtype=float)
    in_vals = pd.to_numeric(pair_df['in_val'], errors='coerce').to_numpy(dtype=float)
    out_vals = pd.to_numeric(pair_df['out_val'], errors='coerce').to_numpy(dtype=float)
    mask = np.isfinite(in_vals) & np.isfinite(out_vals)
    return in_vals[mask], out_vals[mask]


def _draw_group_pair(ax, in_vals, out_vals, xin, xout):
    pfsp._draw_boxplot(ax, [in_vals, out_vals], [xin, xout], [inside_color, outside_color])
    pfsp._overlay_pair_lines_points(ax, in_vals, out_vals, xin, xout, inside_color, outside_color)


def _annotate_panel(ax, stats_rows, *, shared_height=False):
    if not stats_rows:
        return
    vals = []
    for collection in ax.collections:
        offsets = collection.get_offsets()
        if hasattr(offsets, 'shape') and offsets.shape[0] > 0:
            vals.extend(list(np.asarray(offsets[:, 1], dtype=float)))
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size <= 0:
        return
    y_max = float(np.nanmax(vals))
    y_min = float(np.nanmin(vals))
    y_rng = max(1e-9, y_max - y_min)
    y_scale_abs = max(1.0, abs(y_max), abs(y_min))
    base = y_max + 0.14 * y_rng
    h = max(0.05 * y_rng, 0.02 * y_scale_abs)
    if shared_height:
        for row in sorted(stats_rows, key=lambda item: (float(item['x1']), float(item['x2']))):
            pfsp._draw_bracket(ax, float(row['x1']), float(row['x2']), base, h, str(row['sig_q']), float(style_opts['tick_labelsize']))
        ax.set_ylim(top=base + 1.8 * h)
        return
    pfsp._draw_bracket(ax, float(stats_rows[0]['x1']), float(stats_rows[0]['x2']), base, h, str(stats_rows[0]['sig_q']), float(style_opts['tick_labelsize']))
    ax.set_ylim(top=base + 1.8 * h)


fig, axes = plt.subplots(
    1,
    2,
    figsize=(6, 1.25),
    dpi=180,
    constrained_layout=True,
    gridspec_kw={'width_ratios': [3.0, 5.0]},
)
all_stats = []

panel_specs = [
    {
        'panel': 'mrl',
        'title': 'MRL',
        'ylabel': 'MRL',
        'shared_height': True,
        'groups': [
            ('All', _metric_region_frame('all_mrl_overall'), 'all_in_vs_out'),
            ('SS', _metric_region_frame('ss_mrl_overall'), 'ss_in_vs_out'),
            ('CS', _metric_region_frame('cs_mrl_overall'), 'cs_in_vs_out'),
        ],
    },
    {
        'panel': 'ds',
        'title': 'DS',
        'ylabel': 'DS',
        'shared_height': True,
        'groups': [
            ('All', _ds_region_frame('all', slow_use_abs=True), 'all_in_vs_out'),
            ('SS', _ds_region_frame('ss', slow_use_abs=True), 'ss_in_vs_out'),
            ('CS', _ds_region_frame('cs', slow_use_abs=True), 'cs_in_vs_out'),
            ('Theta', _ds_region_frame('theta', slow_use_abs=False), 'theta_ds_in_vs_out'),
            ('Slow Vm', _ds_region_frame('slow', slow_use_abs=True, abs_ds=True), 'slow_ds_in_vs_out'),
        ],
    },
]

for ax, spec in zip(axes, panel_specs):
    panel_rows = []
    xticks = []
    xlabels = []
    for idx, (label, pair_df, comparison_name) in enumerate(spec['groups']):
        xin = float(idx * 2)
        xout = float(idx * 2 + 1)
        in_vals, out_vals = _paired_values(pair_df)
        _draw_group_pair(ax, in_vals, out_vals, xin, xout)
        p_raw, statistic, test_name, n_pairs, shapiro_p = pfsp._paired_test_auto(in_vals, out_vals)
        panel_rows.append({
            'panel': spec['panel'],
            'comparison': comparison_name,
            'n': int(n_pairs),
            'test': test_name,
            'statistic': statistic,
            'shapiro_p': shapiro_p,
            'p_raw': p_raw,
            'x1': xin,
            'x2': xout,
        })
        xticks.extend([xin, xout])
        xlabels.extend([f'{label}\nIn', f'{label}\nOut'])

    ax.set_xticks(xticks)
    ax.set_xticklabels(xlabels, fontsize=float(style_opts['tick_labelsize']))
    if xticks:
        ax.set_xlim(min(xticks) - 0.6, max(xticks) + 0.6)
    ax.set_ylabel(spec['ylabel'], fontsize=float(style_opts['axes_labelsize']))
    ax.set_title(spec['title'], fontsize=float(style_opts['axes_titlesize']))
    pfsp._finalize_axis(ax, style_opts)
    all_stats.extend(panel_rows)

stats_df = pd.DataFrame(all_stats, columns=['panel', 'comparison', 'n', 'test', 'statistic', 'shapiro_p', 'p_raw', 'x1', 'x2'])
stats_df['q_bh_fdr'] = pfsp._bh_fdr(stats_df['p_raw'].to_numpy(dtype=float))
stats_df['sig_q'] = [pfsp._sig_label(v) for v in stats_df['q_bh_fdr'].to_numpy(dtype=float)]

for ax, spec in zip(axes, panel_specs):
    panel_rows = stats_df.loc[stats_df['panel'] == spec['panel']].to_dict('records')
    _annotate_panel(ax, panel_rows, shared_height=bool(spec['shared_height']))

fig.suptitle('CS+ EB-tuned: inside vs outside combined PF', fontsize=float(style_opts['axes_titlesize']) + 1.0, y=1.05)
fig_path = summary_dir / f'pf_split_csplus_eb_tuned_inside_vs_outside_combined_{ANY_PASS_SUFFIX}.svg'
stats_path = summary_dir / f'pf_split_csplus_eb_tuned_inside_vs_outside_combined_stats_{ANY_PASS_SUFFIX}.csv'
fig.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

stats_df = stats_df[['panel', 'comparison', 'n', 'test', 'statistic', 'shapiro_p', 'p_raw', 'q_bh_fdr', 'sig_q']]
stats_df.to_csv(stats_path, index=False)

print('Saved inside-vs-outside PF figure:', Path(fig_path).resolve())
print('Saved inside-vs-outside PF stats CSV:', Path(stats_path).resolve())
print(stats_df.to_string(index=False))


Saved inside-vs-outside PF figure: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/summary_stats/pf_split_csplus_eb_tuned_inside_vs_outside_combined_any100.svg
Saved inside-vs-outside PF stats CSV: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/summary_stats/pf_split_csplus_eb_tuned_inside_vs_outside_combined_stats_any100.csv
panel         comparison  n          test  statistic  shapiro_p    p_raw  q_bh_fdr sig_q
  mrl      all_in_vs_out  7 paired t-test  -5.153903   0.544352 0.002107  0.016858     *
  mrl       ss_in_vs_out  6 paired t-test  -4.315024   0.851174 0.007606  0.022757     *
  mrl       cs_in_vs_out  7 paired t-test  -3.842510   0.487203 0.008534  0.022757     *
   ds      all_in_vs_out  7      wilcoxon   7.000000   0.015525 0.296875  0.440515  n.s.
   ds   

/var/folders/8g/cpkd0cfs4gn915zwbfdfgf9w0000gn/T/ipykernel_13282/2787097335.py:218: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
import importlib
importlib.reload(pfsp)

# Combined CS+ vs CS- (combined / outside / all-bins), first 7 columns
from pathlib import Path
pf_split_csv = OUT_DIR / 'per_cell_summary' / f'egocentric_pf_split_stats_{ANY_PASS_SUFFIX}_3spike.csv'
summary_dir = OUT_DIR / 'summary_stats'
summary_dir.mkdir(parents=True, exist_ok=True)

combined_result = pfsp.plot_pf_split_combined_csplus_vs_csminus_row3(
    pf_split_csv=pf_split_csv,
    out_dir=summary_dir,
    any_pass_threshold=ANY_PASS_THRESHOLD,
    any_pass_suffix=ANY_PASS_SUFFIX,
    regions=['combined', 'outside_combined', 'all_bins'],
    csplus_group='both',
    style_opts={
        'font_family': 'Arial',
        'font_size': 6.0,
        'axes_labelsize': 6.0,
        'axes_titlesize': 6.0,
        'tick_labelsize': 5.0,
        'legend_fontsize': 5.0,
        'axes_linewidth': 0.5,
        'remove_top_right_spines': True,
        'show_p_after_sig': False,
        'show_only_significant': False,
        'group_width_in': group_width_in_combined,
    },
    show=True,
)

if 'results' in combined_result:
    for mode, res in combined_result['results'].items():
        print(f"[{mode}] figure path:", Path(res['figure']).resolve())
        print(f"[{mode}] stats CSV:", Path(res['stats_csv']).resolve())
else:
    print('Saved combined figure path:', Path(combined_result['figure']).resolve())
    print('Saved combined stats CSV:', Path(combined_result['stats_csv']).resolve())

/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/notebooks_HPC/pf_split_group_plotting.py:1112: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


[eb_tuned] figure path: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/summary_stats/pf_split_csplus_eb_tuned_vs_csminus_combined_outside_combined_all_bins_first10_any100.svg
[eb_tuned] stats CSV: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/summary_stats/pf_split_csplus_eb_tuned_vs_csminus_combined_outside_combined_all_bins_first10_stats_any100.csv
[all] figure path: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/summary_stats/pf_split_csplus_all_vs_csminus_combined_outside_combined_all_bins_first10_any100.svg
[all] stats CSV: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any1

/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/notebooks_HPC/pf_split_group_plotting.py:1112: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
import importlib
import miniVI_PlaceCell_analysis_V4.notebooks_HPC.pf_split_group_plotting as pfsp
importlib.reload(pfsp)
import miniVI_PlaceCell_analysis_V4.notebooks_HPC.run_egocentric_plot_cells_head_three_spike_any99 as head_runner
importlib.reload(head_runner)


<module 'miniVI_PlaceCell_analysis_V4.notebooks_HPC.run_egocentric_plot_cells_head_three_spike_any99' from '/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/notebooks_HPC/run_egocentric_plot_cells_head_three_spike_any99.py'>

In [12]:
import importlib
importlib.reload(pfsp)

# Combined CS+ vs CS- direction selectivity (DS), first 6 columns (DS merged + MRL comparison)
from pathlib import Path
pf_split_csv = OUT_DIR / 'per_cell_summary' / f'egocentric_pf_split_stats_{ANY_PASS_SUFFIX}_3spike.csv'
summary_dir = OUT_DIR / 'summary_stats'
summary_dir.mkdir(parents=True, exist_ok=True)

ds_result = pfsp.plot_pf_split_direction_selectivity_csplus_vs_csminus_first4(
    pf_split_csv=pf_split_csv,
    out_dir=summary_dir,
    any_pass_threshold=ANY_PASS_THRESHOLD,
    any_pass_suffix=ANY_PASS_SUFFIX,
    regions=['combined', 'outside_combined', 'all_bins'],
    csplus_group='both',
    style_opts={
        'font_family': 'Arial',
        'font_size': 6.0,
        'axes_labelsize': 6.0,
        'axes_titlesize': 6.0,
        'tick_labelsize': 5.0,
        'legend_fontsize': 5.0,
        'axes_linewidth': 0.5,
        'remove_top_right_spines': True,
        'show_p_after_sig': False,
        'show_only_significant': False,
        'group_width_in': group_width_in_ds,
    },
    show=True,
)

if 'results' in ds_result:
    for mode, res in ds_result['results'].items():
        print(f"[{mode}] DS figure path:", Path(res['figure']).resolve())
        print(f"[{mode}] DS stats CSV:", Path(res['stats_csv']).resolve())
else:
    print('Saved DS figure path:', Path(ds_result['figure']).resolve())
    print('Saved DS stats CSV:', Path(ds_result['stats_csv']).resolve())

/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/notebooks_HPC/pf_split_group_plotting.py:1668: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


[eb_tuned] DS figure path: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/summary_stats/pf_split_ds_csplus_eb_tuned_vs_csminus_combined_outside_combined_all_bins_first9_any100.svg
[eb_tuned] DS stats CSV: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/summary_stats/pf_split_ds_csplus_eb_tuned_vs_csminus_combined_outside_combined_all_bins_first9_stats_any100.csv
[all] DS figure path: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/summary_stats/pf_split_ds_csplus_all_vs_csminus_combined_outside_combined_all_bins_first9_any100.svg
[all] DS stats CSV: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpen

/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/notebooks_HPC/pf_split_group_plotting.py:1668: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## DRZ Preferred vs Non-Preferred Traversals (CS+ EB-tuned)

Trial-by-trial DRZ analysis for **CS+ EB-tuned** cells selected from `per_cell_summary/pass_any99/CSplus` SVGs.
Traversal classes are **P** (preferred) and **NP** (non-preferred), using green-bin logic from column 8 (row 1, all-spike empirical panel).


In [ ]:

import importlib
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd

_top_repo = str(BASE_DIR.parent.resolve())
sys.path[:] = [p for p in sys.path if os.path.normpath(p) != os.path.normpath(_top_repo)]
sys.path.insert(0, str(BASE_DIR.resolve()))
os.environ['PYTHONPATH'] = str(BASE_DIR.resolve())

import notebooks_HPC.egocentric_refined_config as refined_cfg
import utils.placecell_pipeline as _pcp
from utils.spatial_heatmaps import classify_spatial_cells

importlib.reload(refined_cfg)
importlib.reload(_pcp)

preferred_fraction_threshold = 0.5
preferred_half_width_deg = 50.0
min_covered_green_bins = 1
classification_drz_segment_mode = 'all'
classification_phase_signed_drz_filter = True

config = refined_cfg.build_refined_config(
    project_root=BASE_DIR,
    data_root=DATA_ROOT,
    figures_root=FIGURES_ROOT,
    force_recompute=False,
)
config.merged_data_filename = RUNTIME_MERGED_DATA_FILENAME

print('DRZ refined animals:', len(config.animals), tuple(config.animals))
print('DRZ runtime data filename:', config.merged_data_filename)
print('DRZ notebooks root:', config.notebooks_root)

spatial_data = classify_spatial_cells(
    data_folder=str(SPATIAL_CACHE_ROOT),
    folders=config.animals,
    cb_num_threshold=config.pooled.cb_num_threshold,
    cs_peak_rate_threshold=config.pooled.cs_peak_rate_threshold,
    cs_plc_definition_mode=config.pooled.cs_plc_definition_mode,
    snr_threshold=config.analysis.snr_threshold,
)

drz_params = _pcp.PFDRZParams(
    trial_distance_window_cm=20.0,
    trial_detection_window_cm=6.0,
    trial_clip_mode="pf_entry_exit",
    drz_window=1.0,
    drz_bin=0.1,
    distance_mode="euclidean_to_peak",
    min_traversals=10,
    min_traversals_per_type=5,
    subtract_pre_traversal_baseline=False,
    exclude_trials_with_bad_frames=True,
    pf_component_selection="peak_rate",
    max_pf_distance_cm=8.0,
    smooth_window=0.0,
    center_vicinity_min_cm=1,
    center_vicinity_max_cm=5,
    moving_speed_threshold=float(config.analysis.speed_threshold),
    resting_speed_threshold=0.5,
    include_long_cb_as_plateau=True,
    include_resting_plateaus=True,
    phase_signed_drz_filter=True,
    cb_plateau_min_duration_ms=150.0,
    first_n_minutes=first_n_minutes,
)

drz_trial_plot_dilation_bins = 4
drz_trial_plot_dilation_shape = config.place_cell.pf_reliability_dilation_shape

drz_pnp_out_dir = OUT_DIR / "drz_preferred_nonpreferred"
drz_pnp_result = _pcp.generate_drz_trials_and_dataset_egocentric_preferred_nonpreferred(
    config=config,
    spatial_data=spatial_data,
    drz_params=drz_params,
    egocentric_base_dir=ROOT,
    csplus_svg_dir=OUT_DIR / "per_cell_summary" / f"pass_{ANY_PASS_SUFFIX}" / "CSplus",
    figure_save_folder=drz_pnp_out_dir,
    preferred_fraction_threshold=preferred_fraction_threshold,
    preferred_half_width_deg=preferred_half_width_deg,
    min_covered_green_bins=min_covered_green_bins,
    classification_frame_selection_mode='nonrest',
    classification_drz_segment_mode=classification_drz_segment_mode,
    classification_phase_signed_drz_filter=classification_phase_signed_drz_filter,
    pf_reliability_dilation_bins=drz_trial_plot_dilation_bins,
    pf_reliability_dilation_shape=drz_trial_plot_dilation_shape,
    plot_figures=True,
    show_plots=False,
    save_summary_csv=True,
    debug_mode=False,
)

drz_pnp_dataset = drz_pnp_result.get("dataset", {})
drz_pnp_summary = drz_pnp_result.get("plot_summary", {})
print("Output dir:", drz_pnp_summary.get("output_dir"))
print("Summary CSV:", drz_pnp_summary.get("summary_csv"))
print("Selected cells:", drz_pnp_summary.get("n_selected_cells"))


In [ ]:
from pathlib import Path
import pandas as pd
from pandas.errors import EmptyDataError

summary_csv = Path(drz_pnp_summary.get("summary_csv", ""))
out_dir = Path(drz_pnp_summary.get("output_dir", ""))
if not summary_csv.exists():
    raise FileNotFoundError(f"Missing summary CSV: {summary_csv}")
if not out_dir.exists():
    raise FileNotFoundError(f"Missing output dir: {out_dir}")

try:
    df = pd.read_csv(summary_csv)
except EmptyDataError:
    df = pd.DataFrame()

print("Summary rows:", len(df))
if len(df) > 0:
    print(df["status"].value_counts(dropna=False))
    show_cols = [
        "animal_id", "cell_idx", "pf_rank", "status",
        "n_detected_traversals", "n_classified_traversals",
        "n_preferred_traversals", "n_nonpreferred_traversals", "n_unclassified_traversals",
    ]
    show_cols = [c for c in show_cols if c in df.columns]
    print(df[show_cols].head(20).to_string(index=False))
else:
    print('Summary CSV is empty (no rows).')

svg_files = sorted(out_dir.rglob("*.svg"))
png_files = sorted(out_dir.rglob("*.png"))
print("SVG files:", len(svg_files))
print("PNG files:", len(png_files))
if len(png_files) > 0:
    raise AssertionError("Expected SVG-only outputs for DRZ P/NP analysis.")

selected_svg_cells = sorted((OUT_DIR / "per_cell_summary" / f"pass_{ANY_PASS_SUFFIX}" / "CSplus").glob("*.svg"))
print("Selected CSplus SVG cells:", len(selected_svg_cells))
print("Summary CSV:", summary_csv)
print("Output folder:", out_dir)


## DRZ Compare Directions (Preferred vs Non-preferred)

Run the DRZ compare-directions workflow using explicit `all/p/np` channels from `drz_pnp_dataset`.

In [18]:
import importlib
from pathlib import Path
import pandas as pd

import utils.placecell_pipeline as _pcp
importlib.reload(_pcp)

if not isinstance(drz_pnp_dataset, dict) or len(drz_pnp_dataset) == 0:
    raise RuntimeError('drz_pnp_dataset is missing. Run the DRZ P/NP generation cell first.')

PNP_DIRECTION_KEYS = ('all', 'p', 'np')
PNP_DIRECTION_TITLES = ('All', 'Preferred', 'Non-preferred')
drz_direction_avg_mode_pnp = 's1'
drz_smooth_window_pnp = 0.3
drz_avg_smooth_window_pnp = 0.0

drz_window = float(drz_pnp_dataset.get('metadata', {}).get('drz_window', 1.0))
drz_display_xlim_pnp = (-drz_window, drz_window)

pnp_compare_root = OUT_DIR / 'drz_compare_directions_pnp'
pnp_heatmap_root = pnp_compare_root / 'per_cell_heatmaps'
pnp_average_export_subdir = 'pf_drz_pnp_2directions_average_exports'
pnp_average_export_dir = pnp_heatmap_root / pnp_average_export_subdir

print('PNP direction keys:', PNP_DIRECTION_KEYS)
print('Avg mode:', drz_direction_avg_mode_pnp)
print('Smooth window (cm):', drz_smooth_window_pnp)
print('Avg smooth window (cm):', drz_avg_smooth_window_pnp)
print('Compare root:', pnp_compare_root)
print('Heatmap root:', pnp_heatmap_root)
print('Average export dir:', pnp_average_export_dir)


PNP direction keys: ('all', 'p', 'np')
Avg mode: s1
Smooth window (cm): 0.3
Avg smooth window (cm): 0.0
Compare root: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/drz_compare_directions_pnp
Heatmap root: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/drz_compare_directions_pnp/per_cell_heatmaps
Average export dir: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/drz_compare_directions_pnp/per_cell_heatmaps/pf_drz_pnp_2directions_average_exports


### Heatmap

In [19]:
drz_pnp_heatmap_summary_2directions = _pcp.generate_pf_drz_component_heatmaps_2directions(
    dataset=drz_pnp_dataset,
    figure_save_folder=pnp_heatmap_root,
    clear_output=True,
    smooth_window=float(drz_smooth_window_pnp),
    fr_vmax_scale=0.2,
    theta_vlim=(0, 0.4),
    slow_vlim=(-0.4, 0.4),
    fig_width=3.0,
    trial_height=0.01,
    xlim=drz_display_xlim_pnp,
    merge_ss_cs=True,
    merge_plateau=True,
    smooth_plateau=False,
    plateau_vlim=0.01,
    avg_smooth_window=float(drz_avg_smooth_window_pnp),
    show_session_gap_line=True,
    avg_mode=drz_direction_avg_mode_pnp,
    save_average_exports=True,
    average_export_subdir=pnp_average_export_subdir,
    output_prefix='pf_drz_pnp_2directions',
    direction_keys=PNP_DIRECTION_KEYS,
    direction_titles=PNP_DIRECTION_TITLES,
    show_plot=False,
)

for cat, info in drz_pnp_heatmap_summary_2directions.items():
    print(f'{cat} summary (P/NP 2 directions): {info}')


/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/utils/placecell_pipeline.py:8650: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  tight_fn()
/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/utils/placecell_pipeline.py:8544: RuntimeWarning: Mean of empty slice
  mean_trace = _smooth_avg(np.nanmean(stack, axis=0))


[CSplus] 1/7 saved: file:///Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/drz_compare_directions_pnp/per_cell_heatmaps/pf_drz_pnp_2directions_CSplus_heatmaps/CKII_pAce21_PR_20250806_Cell3_heatmap_2directions.svg


/opt/homebrew/Caskroom/miniforge/base/envs/AdamLab/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/utils/placecell_pipeline.py:8474: RuntimeWarning: Mean of empty slice
  ss_mean = _smooth_avg(np.nanmean(ss_stack, axis=0))
/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/utils/placecell_pipeline.py:8483: RuntimeWarning: Mean of empty slice
  cs_mean = _smooth_avg(np.nanmean(cs_stack, axis=0))


[CSplus] 2/7 saved: file:///Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/drz_compare_directions_pnp/per_cell_heatmaps/pf_drz_pnp_2directions_CSplus_heatmaps/CKII_pAce38_PX_20251126_Cell1_heatmap_2directions.svg
[CSplus] 3/7 saved: file:///Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/drz_compare_directions_pnp/per_cell_heatmaps/pf_drz_pnp_2directions_CSplus_heatmaps/CKII_pAce38_PX_20251126_Cell6_heatmap_2directions.svg
[CSplus] 4/7 saved: file:///Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/drz_compare_directions_pnp/per_cell_heatmaps/pf_drz_pnp_2directions_CSplus_heatmaps/CKII_pAce38_PX_20251126_Cell7_heatmap_2directions.svg
[CSplus] 5/7 saved: file:///Users/

/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/utils/placecell_pipeline.py:23557: RuntimeWarning: Mean of empty slice
  


### Average Plot

In [20]:
cat = 'CSplus'
out_path = pnp_compare_root / 'pf_drz_CSplus_average_primary_secondary_pnp_2directions.svg'
fig = _pcp.plot_pf_drz_category_primary_secondary_from_dataset_2directions(
    dataset=drz_pnp_dataset,
    category=cat,
    title='CS+ PLCs (Preferred vs Non-preferred)',
    out_path=out_path,
    smooth_window=float(drz_smooth_window_pnp),
    avg_smooth_window=float(drz_avg_smooth_window_pnp),
    plot_trials=True,
    trial_alpha=0.2,
    trial_linewidth=0.5,
    theta_ylim=(0, 0.2),
    slow_ylim=(-0.05, 0.2),
    fr_ylim_scale=1.2,
    ss_cs_ylim_scale=1.2,
    figsize_per_block=(3.4, 2.4),
    xlim=drz_display_xlim_pnp,
    fr_row_ylim_max=1.5,
    ss_cs_row_ylim_max=1.1,
    min_traversals_per_type=int(drz_params.min_traversals_per_type),
    use_saved_average_exports=True,
    average_export_dir=pnp_average_export_dir,
    avg_mode=drz_direction_avg_mode_pnp,
    enforce_min_traversals_on_saved_exports=True,
    direction_keys=PNP_DIRECTION_KEYS,
    direction_titles=PNP_DIRECTION_TITLES,
    explicit_preferred_nonpreferred_keys=('p', 'np'),
    show_plot=True,
)
print('Generated:', out_path)
_ = fig


PF1 counts: All (n=4), Pref (n=4), Non-pref (n=4)

/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/utils/placecell_pipeline.py:22699: RuntimeWarning: Mean of empty slice
  
/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/utils/placecell_pipeline.py:22700: RuntimeWarning: Mean of empty slice
  def _load_drz_dataset_from_2direction_exports(
/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/utils/placecell_pipeline.py:9107: RuntimeWarning: Mean of empty slice
  trace = np.nanmean(stack, axis=0)



PF2 counts: All (n=4), Pref (n=4), Non-pref (n=4)
Generated: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/drz_compare_directions_pnp/pf_drz_CSplus_average_primary_secondary_pnp_2directions.svg


/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/utils/placecell_pipeline.py:23694: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  


In [21]:
drz_pnp_directionality_results = _pcp.compute_pf_drz_pref_nonpref_stats(
    drz_pnp_dataset,
    timepoint_cm=0.0,
    min_traversals_per_type=int(drz_params.min_traversals_per_type),
    smooth_window_cm=0.0,
    auc_window_cm=None,
    slow_auc_window_cm=(-0.5, 0.5),
    peak_window_cm=(-0.5, 0.5),
    use_saved_average_exports=True,
    average_export_dir=pnp_average_export_dir,
    avg_mode=drz_direction_avg_mode_pnp,
    direction_keys=PNP_DIRECTION_KEYS,
    explicit_preferred_nonpreferred_keys=('p', 'np'),
)
print('Skipped counts:', drz_pnp_directionality_results.get('skipped_counts', {}))

drz_pnp_directionality_plot_summary = _pcp.plot_pf_drz_pref_nonpref_stats(
    drz_pnp_directionality_results,
    figure_save_folder=pnp_compare_root,
    show_plot=True,
    save_csv=True,
)
print('DRZ P/NP directionality outputs:')
for key, value in drz_pnp_directionality_plot_summary.items():
    print(f'  {key}: {value}')


[pref_nonpref_stability] No NaN metrics from flat/invalid traces after inclusion gating.
Skipped counts: {'missing': 0, 'low_traversals': 6, 'no_preferred': 0}
DRZ P/NP directionality outputs:
  figure: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/drz_compare_directions_pnp/pf_drz_pref_nonpref_summary_5panels.svg
  per_cell_csv: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/drz_compare_directions_pnp/pf_drz_pref_nonpref_per_cell.csv
  stats_csv: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter_full/head_any100_three_spike_compare/drz_compare_directions_pnp/pf_drz_pref_nonpref_stats.csv
  figure_2subplots: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/e

/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/utils/placecell_pipeline.py:15861: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/utils/placecell_pipeline.py:16170: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
if not pnp_compare_root.exists():
    raise FileNotFoundError(f'Missing compare output dir: {pnp_compare_root}')

heatmap_svgs = sorted((pnp_heatmap_root).rglob('*.svg'))
avg_fig = pnp_compare_root / 'pf_drz_CSplus_average_primary_secondary_pnp_2directions.svg'
stats_csv = Path(drz_pnp_directionality_plot_summary.get('stats_csv', ''))
per_cell_csv = Path(drz_pnp_directionality_plot_summary.get('per_cell_csv', ''))
stats_fig = Path(drz_pnp_directionality_plot_summary.get('figure', ''))
stats_fig_sscs = Path(drz_pnp_directionality_plot_summary.get('figure_ss_cs', ''))

print('CSplus selected cells from SVG folder:', len(list((OUT_DIR / 'per_cell_summary' / f'pass_{ANY_PASS_SUFFIX}' / 'CSplus').glob('*.svg'))))
print('Heatmap SVG count:', len(heatmap_svgs))
print('Average figure exists:', avg_fig.exists(), avg_fig)
print('Stats figure exists:', stats_fig.exists(), stats_fig)
print('Stats SS/CS figure exists:', stats_fig_sscs.exists(), stats_fig_sscs)
print('Per-cell CSV exists:', per_cell_csv.exists(), per_cell_csv)
print('Stats CSV exists:', stats_csv.exists(), stats_csv)

if not avg_fig.exists():
    raise FileNotFoundError(f'Missing average figure: {avg_fig}')
if not per_cell_csv.exists() or not stats_csv.exists():
    raise FileNotFoundError('Missing directionality CSV outputs.')

from pandas.errors import EmptyDataError
try:
    per_cell_df = pd.read_csv(per_cell_csv)
except EmptyDataError:
    per_cell_df = pd.DataFrame()

print('Per-cell stats rows:', len(per_cell_df))
if len(per_cell_df) > 0:
    print('direction_pref values:', sorted(per_cell_df['direction_pref'].dropna().astype(str).unique().tolist()))
    print('direction_nonpref values:', sorted(per_cell_df['direction_nonpref'].dropna().astype(str).unique().tolist()))

summary_csv = Path(drz_pnp_summary.get('summary_csv', ''))
if summary_csv.exists():
    try:
        summary_df = pd.read_csv(summary_csv)
    except EmptyDataError:
        summary_df = pd.DataFrame()
    if not summary_df.empty:
        cols = [c for c in [
            'n_detected_traversals', 'n_classified_traversals',
            'n_preferred_traversals', 'n_nonpreferred_traversals', 'n_unclassified_traversals'
        ] if c in summary_df.columns]
        if cols:
            print('Traversal totals:', summary_df[cols].sum(numeric_only=True).to_dict())
    else:
        print('Traversal summary CSV is empty.')
